# Selección

Los nueve entrenamientos comparten protocolo —mismo corpus, mismo reparto,
`lr=1e-4`, lote 32, 100 épocas, 5 de calentamiento, paciencia 8, 448 px— así
que una sola tabla los ordena y de ella salen los dos modelos que usa el resto
del trabajo:

- **estudiante**: el mejor macro-F1 que quepa en `PRESUPUESTO_MB`. El producto
  final corre en el dispositivo del usuario, así que el costo que manda es lo
  que el navegador descarga, no los milisegundos en una CPU de escritorio. El
  presupuesto es una decisión de producto, no un óptimo.
- **docente**: el mejor macro-F1 sin restricción de tamaño.

Macro-F1 y no accuracy porque el corpus está desbalanceado. Se selecciona por
validación y se reporta sobre test.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/CCSandoval/ButterflyModeling.git"

if Path("/kaggle").exists():
    CLON = "/tmp/ButterflyModeling"
    if not Path(CLON).exists():                     # el repo es público: clonar no pide credenciales
        subprocess.run(["git", "clone", "--depth", "1", REPO, CLON], check=True)
    sys.path.insert(0, CLON)
    os.chdir(CLON)
    print("clon en", CLON, "·", subprocess.run(
        ["git", "log", "-1", "--format=%h %s"], cwd=CLON,
        capture_output=True, text=True).stdout.strip())

    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["GITHUB_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        print("sin GITHUB_TOKEN: no se publicará")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

In [ ]:
import pandas as pd

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.vizStyle import INK_MUTED, SERIE_1, SERIE_2, SERIE_3, applyStyle

PRESUPUESTO_MB = 8.0        # float32, lo que descarga el navegador
CORPUS = corpus.manifiesto()["creado"]

# clave (arquitectura, descongelado): congelado y fine-tuning son regímenes
# distintos y ambos deben verse en la tabla, no solo el más reciente
runPorArquitectura = {}
for run in registro.listRuns():
    clave = (run.get("architecture"), run.get("finetuning", 0.0))
    if (run.get("role") != "destilado" and run.get("dataset") == CORPUS
            and clave not in runPorArquitectura):
        runPorArquitectura[clave] = run

print(f"corpus {CORPUS[:10]} · {len(runPorArquitectura)} runs")

## 1 · Calidad y costo

`pesos_MB` es lo que realmente se descarga: parámetros x 4 bytes. El archivo
`.keras` no sirve como medida — incluye el estado del optimizador de Adam, que
casi triplica el tamaño y no se despliega.

In [ ]:
filas = []
for (arquitectura, descongelado), run in runPorArquitectura.items():
    modelo = registro.modeloInferencia(run["run_id"])
    parametros = modelo.count_params()
    metricas = registro.loadRunMetrics(run["run_id"])["splits"]["test"]
    latencia, desviacion = evalStats.medirLatencia(modelo)
    filas.append({
        "arquitectura": arquitectura,
        "descongelado": descongelado,
        "validate_macro_f1": run["validate_macro_f1"],
        "test_macro_f1": run["test_macro_f1"],
        "test_accuracy": run["test_accuracy"],
        "top3": metricas["top3_accuracy"],
        "parametros_M": round(parametros / 1e6, 2),
        "pesos_MB": round(parametros * 4 / 1e6, 1),
        "int8_MB": round(parametros / 1e6, 1),
        "latencia_ms": round(latencia, 1),
        "latencia_sd_ms": round(desviacion, 1),
        "minutos_entrenamiento": run["training_minutes"],
        "run_id": run["run_id"],
    })

comparacion = pd.DataFrame(filas).sort_values("validate_macro_f1", ascending=False).reset_index(drop=True)
comparacion.drop(columns="run_id")

## 2 · Accuracy vs macro-F1

La distancia entre las dos barras mide cuánto de la accuracy viene de las
especies abundantes.

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

orden = comparacion.sort_values("validate_macro_f1")
y = range(len(orden))

fig, ax = plt.subplots(figsize=(8, 0.6 * len(orden) + 1.5))
ax.barh([i - 0.18 for i in y], orden["test_accuracy"], height=0.32, color=SERIE_1, label="accuracy")
ax.barh([i + 0.18 for i in y], orden["test_macro_f1"], height=0.32, color=SERIE_2, label="macro F1")
ax.set_yticks(list(y), orden["arquitectura"])
ax.set(xlim=(0, 1), xlabel="Test", title="Las nueve arquitecturas")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 3 · Calidad vs descarga

La latencia se reporta como referencia, pero se midió en CPU de escritorio: en
un navegador sobre WebGL/WASM y un teléfono de gama media es varias veces
mayor. La descarga, en cambio, es la misma en todas partes.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
dentro = comparacion["pesos_MB"] <= PRESUPUESTO_MB

ax.axvspan(0, PRESUPUESTO_MB, color=SERIE_3, alpha=0.10)
ax.axvline(PRESUPUESTO_MB, color=INK_MUTED, linestyle="--", linewidth=1.2)
ax.text(PRESUPUESTO_MB, comparacion["test_macro_f1"].min(), f" presupuesto {PRESUPUESTO_MB:.0f} MB",
        fontsize=9, va="bottom", color=INK_MUTED)
ax.scatter(comparacion.loc[dentro, "pesos_MB"], comparacion.loc[dentro, "test_macro_f1"],
           s=90, color=SERIE_1, label="cabe", zorder=3)
ax.scatter(comparacion.loc[~dentro, "pesos_MB"], comparacion.loc[~dentro, "test_macro_f1"],
           s=90, color=SERIE_2, label="excede", zorder=3)
for _, f in comparacion.iterrows():
    ax.annotate(f["arquitectura"], (f["pesos_MB"], f["test_macro_f1"]),
                textcoords="offset points", xytext=(8, 5), fontsize=8)
ax.set(xlabel="Pesos que descarga el navegador (MB, float32)", ylabel="macro F1 (test)",
       title="Calidad vs descarga")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 4 · Estudiante

El que sirve el sitio.

In [ ]:
candidatos = comparacion[comparacion["pesos_MB"] <= PRESUPUESTO_MB]
if candidatos.empty:
    raise ValueError(f"Ningún candidato cabe en {PRESUPUESTO_MB} MB")

estudiante = candidatos.iloc[0]
runEstudiante = runPorArquitectura[(estudiante["arquitectura"], estudiante["descongelado"])]

registro.writeJson(registro.OUTPUTS_DIR / "student.json", {
    "run_id": runEstudiante["run_id"],
    "architecture": estudiante["arquitectura"],
    "finetuning": float(estudiante["descongelado"]),
    "dataset": CORPUS,
    "test_accuracy": float(estudiante["test_accuracy"]),
    "test_macro_f1": float(estudiante["test_macro_f1"]),
    "validate_macro_f1": float(estudiante["validate_macro_f1"]),
    "top3_accuracy": float(estudiante["top3"]),
    "pesos_MB": float(estudiante["pesos_MB"]),
    "latencia_ms": float(estudiante["latencia_ms"]),
    "latencia_sd_ms": float(estudiante["latencia_sd_ms"]),
    "presupuesto_MB": PRESUPUESTO_MB,
})

print(f"Estudiante: {estudiante['arquitectura']} (descongelado {estudiante['descongelado']}) · "
      f"macro-F1 {estudiante['test_macro_f1']:.4f} · top-3 {estudiante['top3']:.4f} · "
      f"{estudiante['pesos_MB']:.1f} MB")

for _, f in comparacion[comparacion["pesos_MB"] > PRESUPUESTO_MB].iterrows():
    if f["validate_macro_f1"] > estudiante["validate_macro_f1"]:
        print(f"  queda fuera {f['arquitectura']}: {f['test_macro_f1'] - estudiante['test_macro_f1']:+.4f} "
              f"de macro-F1 en test y {f['top3'] - estudiante['top3']:+.4f} de top-3, "
              f"pero pesa {f['pesos_MB'] / estudiante['pesos_MB']:.1f}x más")

enInt8 = comparacion[comparacion["int8_MB"] <= PRESUPUESTO_MB]
mejorInt8 = enInt8.iloc[0]
print(f"\ncon el mismo presupuesto medido en int8 entrarían {len(enInt8)} de "
      f"{len(comparacion)} y ganaría {mejorInt8['arquitectura']} "
      f"({mejorInt8['test_macro_f1']:.4f} de macro-F1 en test, "
      f"{mejorInt8['int8_MB']:.1f} MB)")

## 5 · Docente

El techo del pool, y el punto de partida de la destilación. Si el mismo run
gana los dos roles no hay nada que destilar.

In [ ]:
docente = comparacion.iloc[0]
runDocente = runPorArquitectura[(docente["arquitectura"], docente["descongelado"])]

registro.writeJson(registro.OUTPUTS_DIR / "teacher.json", {
    "run_id": runDocente["run_id"],
    "architecture": docente["arquitectura"],
    "dataset": CORPUS,
    "test_accuracy": float(docente["test_accuracy"]),
    "test_macro_f1": float(docente["test_macro_f1"]),
    "validate_macro_f1": float(docente["validate_macro_f1"]),
    "latencia_ms": float(docente["latencia_ms"]),
    "latencia_sd_ms": float(docente["latencia_sd_ms"]),
    "parametros_M": float(docente["parametros_M"]),
})

segundo = comparacion.iloc[1]
print(f"Docente: {docente['arquitectura']} · macro-F1 {docente['test_macro_f1']:.4f}")
print(f"Ventaja sobre el siguiente ({segundo['arquitectura']}): "
      f"{docente['test_macro_f1'] - segundo['test_macro_f1']:+.4f}")

brecha = docente["test_macro_f1"] - estudiante["test_macro_f1"]
if runDocente["run_id"] == runEstudiante["run_id"]:
    print("\nel mismo run gana ambos roles: destilar no aplica")
else:
    print(f"\nbrecha con el estudiante {estudiante['arquitectura']}: {brecha:+.4f}")

from scripts.kaggle import publicarArchivos
if Path("/kaggle").exists() and os.environ.get("GITHUB_TOKEN"):
    publicarArchivos(f"Select {estudiante['arquitectura']} as student and {docente['arquitectura']} as teacher",
                     "outputs/student.json", "outputs/teacher.json")

## Resumen

In [ ]:
comparacion.drop(columns='run_id')